<a href="https://colab.research.google.com/github/Trishul32/Product-Recommendation/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries
!pip install pandas numpy scikit-learn surprise

import pandas as pd
import numpy as np

In [ ]:
books = pd.read_csv("https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv")
ratings = pd.read_csv("https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv")

print(books.head())
print(ratings.head())

   book_id  goodreads_book_id  best_book_id  work_id  books_count       isbn  \
0        1            2767052       2767052  2792775          272  439023483   
1        2                  3             3  4640799          491  439554934   
2        3              41865         41865  3212258          226  316015849   
3        4               2657          2657  3275794          487   61120081   
4        5               4671          4671   245494         1356  743273567   

         isbn13                      authors  original_publication_year  \
0  9.780439e+12              Suzanne Collins                     2008.0   
1  9.780440e+12  J.K. Rowling, Mary GrandPré                     1997.0   
2  9.780316e+12              Stephenie Meyer                     2005.0   
3  9.780061e+12                   Harper Lee                     1960.0   
4  9.780743e+12          F. Scott Fitzgerald                     1925.0   

                             original_title  ... ratings_count  \
0 

PRE-PROCESSING

In [ ]:
# Merge datasets
data = pd.merge(ratings, books, left_on='book_id', right_on='book_id')

# Reduce size for faster processing
data = data[['user_id', 'book_id', 'title', 'authors', 'average_rating']]
data = data.sample(20000, random_state=42)

data.head()

,user_id,book_id,title,authors,average_rating
3623535,42562,2757,"Ahab's Wife, or The Star-Gazer",Sena Jeter Naslund,4.02
3985638,43232,134,"City of Glass (The Mortal Instruments, #3)",Cassandra Clare,4.34
2983642,37244,1463,"Enchanters' End Game (The Belgariad, #5)",David Eddings,4.17
5812251,53366,71,Frankenstein,"Mary Wollstonecraft Shelley, Percy Bysshe Shel...",3.75
2208852,29634,3339,"The Atlantis Complex (Artemis Fowl, #7)",Eoin Colfer,3.97


In [ ]:
!pip install numpy==1.26.4
!pip install scikit-surprise --no-cache-dir

In [ ]:
from surprise import Dataset, Reader, KNNBasic
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(data[['user_id', 'book_id', 'average_rating']], reader)

trainset, testset = train_test_split(surprise_data, test_size=0.2)

algo = KNNBasic(sim_options={'name': 'cosine', 'user_based': True})
algo.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [ ]:
algo.predict(uid=1, iid=50)

Prediction(uid=1, iid=50, r_ui=None, est=4.01977, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'})

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

books['content'] = books['title'] + " " + books['authors']

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(books['content'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
indices = pd.Series(books.index, index=books['title']).drop_duplicates()

def content_recommend(title, n=5):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n+1]

    book_indices = [i[0] for i in sim_scores]
    return books['title'].iloc[book_indices]

In [ ]:
def hybrid_recommend(user_id, title, n=5):
    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:20]

    results = []

    for i, score in sim_scores:
        book_id = books.iloc[i]['book_id']

        # collaborative score
        pred = algo.predict(user_id, book_id).est

        # hybrid score
        hybrid_score = (0.5 * score) + (0.5 * pred)

        results.append((books.iloc[i]['title'], hybrid_score))

    results = sorted(results, key=lambda x: x[1], reverse=True)
    return results[:n]

In [ ]:
def explain_recommendation(user_id, title):
    recs = hybrid_recommend(user_id, title, 3)

    explanations = []

    for rec, score in recs:
        explanations.append(
            f"Recommended '{rec}' because:\n"
            f"- It is similar to '{title}' (content similarity)\n"
            f"- Users with similar taste rated it highly (collaborative filtering)\n"
            f"- Final hybrid score: {round(score,2)}\n"
        )

    return "\n".join(explanations)

In [ ]:
from surprise import accuracy

predictions = algo.test(testset)
accuracy.rmse(predictions)

RMSE: 0.2493


0.24930185089630802

In [ ]:
print(content_recommend("The Hobbit"))

print(hybrid_recommend(user_id=1, title="The Hobbit"))

print(explain_recommendation(user_id=1, title="The Hobbit"))

963     J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...
1128     The History of the Hobbit, Part One: Mr. Baggins
2308                                The Children of Húrin
465                             The Hobbit: Graphic Novel
4975         Unfinished Tales of Númenor and Middle-Earth
Name: title, dtype: object
[('J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings', 2.3650301985051465), ('The History of the Hobbit, Part One: Mr. Baggins', 2.291824595639127), ('The Children of Húrin', 2.242660756159359), ('The Hobbit: Graphic Novel', 2.2414875743010847), ('Unfinished Tales of Númenor and Middle-Earth', 2.2214440202455754)]
Recommended 'J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings' because:
- It is similar to 'The Hobbit' (content similarity)
- Users with similar taste rated it highly (collaborative filtering)
- Final hybrid score: 2.37

Recommended 'The History of the Hobbit, Part One: Mr. Baggins' because:
- It is similar to 'The Hobbit' 